# DenseNet（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，DenseNet（Densely Connected Convolutional Network）を用いてCIFAR100データセットの100クラス物体認識を行い，層間を密に接続するDense Blockの構造とその効果について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100はCIFAR10と同じく32×32ピクセルのカラー画像データセットですが，クラス数が100（1クラスあたり学習用500枚，評価用100枚）とより細かいクラス分けになっています．

学習用データに対しては，`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## Dense Block
DenseNetは，各層の出力を後続の全ての層の入力として用いる「Dense Block」という構造を導入したネットワークです．通常のネットワークやResNetのショートカット構造では，ある層の出力は直後の層（あるいは1つ先のブロック）にしか伝わりませんが，DenseNetでは，Dense Block内の層$l$の入力は，それ以前の全ての層の出力を**チャンネル方向に連結（concatenate）**したものになります．

$$x_l = H_l([x_0, x_1, \dots, x_{l-1}])$$

ここで，$[x_0, x_1, \dots, x_{l-1}]$はチャンネル方向の連結を表し，$H_l$はBatch Normalization，ReLU，畳み込みからなる変換です．このように層をまたいで特徴マップを再利用することで，勾配消失を緩和しつつ，少ないパラメータ数で表現力の高いネットワークを構築できます．

### Growth Rate
Dense Block内の各層が新たに出力する特徴マップのチャンネル数を**Growth Rate**（$k$）と呼びます．各層の出力はGrowth Rate分のチャンネル数しか持たず，それ以前の全ての層の出力と連結されるため，Dense Block入力のチャンネル数を$k_0$とすると，Block内の$l$番目の層の入力チャンネル数は$k_0 + k \times (l-1)$となります．Growth Rateを小さく保つことで，Dense Blockが深くなってもパラメータ数の増加を抑えられます．

### Bottleneck構造（DenseNet-B）
各層をそのまま3×3畳み込みだけで構成すると，層を経るごとに入力チャンネル数が増加し，計算コストが大きくなります．そこで，3×3畳み込みの前に1×1畳み込みを挿入し，チャンネル数を`bn_size * growth_rate`（本ノートブックでは`bn_size=4`）に削減してから3×3畳み込みを行うBottleneck構造を導入します．この構造を用いたDenseNetは，元論文で**DenseNet-B**と呼ばれています．

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate, bn_size=4):
        super().__init__()
        inter_channels = bn_size * growth_rate
        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, inter_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(inter_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(inter_channels, growth_rate, kernel_size=3, padding=1, bias=False),
        )

    def forward(self, x):
        out = self.layer(x)
        # 入力と出力をチャンネル方向に連結し，後続の層にそのまま伝える
        return torch.cat([x, out], dim=1)


class DenseBlock(nn.Module):
    def __init__(self, n_layers, in_channels, growth_rate):
        super().__init__()
        layers = []
        for i in range(n_layers):
            # i層目の入力チャンネル数は，直前までの全ての層の出力が連結された分だけ増加する
            layers.append(DenseLayer(in_channels + i * growth_rate, growth_rate))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

## Transition Layer
Dense Block内では特徴マップのチャンネル数が単調に増加していくため，Block内で特徴マップサイズ（解像度）を変更することはできません．そこで，Dense Blockの間には，チャンネル数の削減とダウンサンプリング（プーリング）を行う**Transition Layer**を挿入します．Transition Layerは，1×1畳み込みとAverage Poolingから構成されます．

さらに，1×1畳み込みで出力するチャンネル数を，入力チャンネル数に圧縮率$\theta$（本ノートブックでは$\theta=0.5$）を乗じた数まで削減することで，パラメータ数をさらに抑えることができます．この圧縮を導入したDenseNetは，元論文で**DenseNet-C**と呼ばれ，Bottleneck構造と組み合わせたものは**DenseNet-BC**と呼ばれます．本ノートブックでは，このDenseNet-BC構成を実装します．

In [ ]:
class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )

    def forward(self, x):
        return self.layer(x)

### DenseNetの定義
上で定義したDense BlockとTransition Layerを用いてDenseNet全体を構築します．DenseNetには，224×224ピクセルのImageNet画像を対象とした「ImageNet版」と，32×32ピクセルのCIFAR10/100を対象とした「CIFAR版」があります．本ノートブックでは，元論文でCIFAR実験に使用されたCIFAR版（DenseNet-BC）を実装します．CIFAR版のDenseNetは，最初の畳み込み層（stem），3つのDense Block，2つのTransition Layer，および出力層（全結合層）から構成されます（両者の違いの詳細は本ノートブック末尾の「ImageNet版DenseNetとCIFAR版（本ノートブック）の違い」を参照）．

`depth`は畳み込みの総層数を指定します．Bottleneck構造では1つのDense Layerあたり畳み込みを2つ（1×1と3×3）使用するため，Dense Block 1つあたりのDense Layer数`n_layers`は

$$\text{n\_layers} = \frac{\text{depth} - 4}{6}$$

という関係から逆算しています（3つのDense Blockで`depth - 4`層分の畳み込みを担当し，Bottleneckにより層数が2倍になるため6で割る）．

In [ ]:
class DenseNet(nn.Module):
    def __init__(self, depth=40, growth_rate=12, reduction=0.5, n_class=100):
        super().__init__()
        # 指定した深さ（畳み込みの層数）でネットワークを構築できるかを確認
        assert (depth - 4) % 6 == 0, 'depth should be 6n+4 (e.g. 40, 100).'
        n_layers = (depth - 4) // 6

        n_channels = 2 * growth_rate

        self.conv1 = nn.Conv2d(3, n_channels, kernel_size=3, padding=1, bias=False)

        self.dense1 = DenseBlock(n_layers, n_channels, growth_rate)
        n_channels += n_layers * growth_rate
        out_channels = int(n_channels * reduction)
        self.trans1 = TransitionLayer(n_channels, out_channels)
        n_channels = out_channels

        self.dense2 = DenseBlock(n_layers, n_channels, growth_rate)
        n_channels += n_layers * growth_rate
        out_channels = int(n_channels * reduction)
        self.trans2 = TransitionLayer(n_channels, out_channels)
        n_channels = out_channels

        self.dense3 = DenseBlock(n_layers, n_channels, growth_rate)
        n_channels += n_layers * growth_rate

        self.bn = nn.BatchNorm2d(n_channels)
        self.relu = nn.ReLU(inplace=True)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(n_channels, n_class)

    def forward(self, x):
        x = self.conv1(x)

        x = self.dense1(x)
        x = self.trans1(x)

        x = self.dense2(x)
        x = self.trans2(x)

        x = self.dense3(x)

        x = self.relu(self.bn(x))
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## ネットワークの作成
定義したDenseNetを作成します．`depth`でネットワークの深さ，`growth_rate`でGrowth Rate（$k$）を指定します．`.to(device)`によりモデルのパラメータを`device`上に配置します．

最適化手法にはモーメンタムSGDを用い，学習率0.01，モーメンタム0.9，weight decayを5e-4とします．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
depth = 40
growth_rate = 12

model = DenseNet(depth=depth, growth_rate=growth_rate, n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版DenseNetとCIFAR版（本ノートブック）の違い
このノートブックで実装したDenseNetは，CIFAR100（32×32入力）向けに元論文で提案されたCIFAR用の構成（DenseNet-BC）です．広く使われているImageNet版DenseNetとは，主に次の点が異なります．

| | ImageNet版 | CIFAR版（本ノートブック） |
|---|---|---|
| 入力画像サイズ | 224×224 | 32×32 |
| stem（最初の畳み込み部） | 7×7 conv (stride2) → 3×3 maxpool (stride2) | 3×3 conv (stride1) の1層のみ |
| Dense Blockの数 | 4 | 3 |
| 各Dense Block間のダウンサンプリング回数 | 3回（Transition Layer内のAverage Pooling） | 2回 |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

ImageNet版では入力解像度が大きいため，stemの段階で大きくダウンサンプリングしてからDense Blockに入力しますが，CIFAR版では32×32という小さな入力画像に対してこれ以上早期にダウンサンプリングすると特徴マップが小さくなりすぎるため，stemは3×3 stride1の畳み込み1層のみとし，ダウンサンプリングはTransition Layer内のAverage Poolingのみで行っています．また，`depth`引数を大きくする（例：100）ことで，元論文のより深いCIFAR用構成を再現することも可能です．

## 課題

1. Growth Rate（`growth_rate`）を変更して，パラメータ数や認識精度がどのように変化するか確認しましょう．

2. ネットワークの深さ（`depth`）を変更して，パラメータ数や認識精度，学習にかかる時間がどのように変化するか確認しましょう．

3. Transition Layerの圧縮率$\theta$を変更（あるいは圧縮を行わないDenseNet-B構成に変更）し，パラメータ数と認識精度への影響を確認しましょう．